# Simulation and Modeling Lab: FIA vs IC Rapid Test for Dengue (NS1)
**Question:** Does the Fluorescent Immunoassay (FIA) detect dengue better than the Immunochromatography (IC) rapid test?

**Source paper:** Ruchusatsawat et al., *Scientific Reports* 2022;12:17299 (404 plasma samples, 158 dengue-confirmed by RT-qPCR + ELISA).

## Dataset used: `dengue_FIA_vs_IC_reconstructed.csv` (paper data only)
I compared the three available CSV files and chose this one, for these reasons:

| CSV | Used? | Reason |
|---|---|---|
| `dengue_FIA_vs_IC_reconstructed.csv` | **Yes** | FIA and IC both tested for the **same marker (NS1)**, on patient plasma, against the same reference standard. Equal group sizes (158 vs 158). |
| `dengue_ic_vs_fluorescence.csv` | No | Its fluorescence rows measure **IP-10 in lab-spiked serum**, a different marker and sample type, with only 63 positive samples. Not comparable with IC NS1. |
| `dengue_combined_FIA_IC.csv` | No | Combining would mix NS1 patient data with IP-10 spiked-serum data, so the two groups would no longer be like-for-like. |
| `fia_reader_vs_ic_visual_dengue.csv` | No | Summary table only (no per-sample rows). |

**Data note:** the paper publishes only summary counts (FIA 125/158 and IC 121/158 detected). The rows are rebuilt from those counts, so row order and IDs are arbitrary and the two tests are treated as **unpaired groups**.

## Hypotheses (alpha = 0.05)
* **H0:** FIA and IC have the same sensitivity (detection rate)
* **H1:** FIA sensitivity is higher than IC sensitivity

## Techniques used (all from class notebooks)
| Technique | Class notebook | Role in this lab |
|---|---|---|
| Chi-square test | Day 7 | **Main test.** Compares observed counts (detected / missed) with the counts expected if both tests were equal. |
| Monte Carlo simulation | Day 4 (with seeds from Day 3) | **Cross-check 1.** Simulates thousands of studies under H0 and counts how often chance alone gives a gap this large. |
| Two-sample t-test | Day 6 | **Cross-check 2.** Compares the mean detection rate of the two groups. |

## Import Necessary Libraries

In [ ]:
import random
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
import matplotlib.pyplot as plt

## Step 1: Load the dataset and make the two groups
We keep only the **dengue-positive** samples, because sensitivity asks: "of the truly sick samples, how many did each test detect?" Each row is `1` (test detected dengue) or `0` (test missed it).

In [ ]:
FILE = 'dengue_FIA_vs_IC_reconstructed.csv'
if not os.path.exists(FILE):          # Google Colab: upload the CSV when asked
    from google.colab import files
    files.upload()
df = pd.read_csv(FILE)
print(df.shape)
df.head()

In [ ]:
sick = df[df.True_Status == 'Disease']

data1 = np.array(sick[sick.Test_Type == 'FIA'].Correct)   # Sample 1: FIA
data2 = np.array(sick[sick.Test_Type == 'IC'].Correct)    # Sample 2: IC

n = len(data1)
print('FIA: detected', data1.sum(), 'of', len(data1), '-> sensitivity', round(data1.mean()*100, 2), '%')
print('IC : detected', data2.sum(), 'of', len(data2), '-> sensitivity', round(data2.mean()*100, 2), '%')

---
# Technique 1 (MAIN TEST): Chi-Square Test, from Day 7
**What it does:** compares the **observed** counts with the **expected** counts under H0, using chi-square = sum of (O - E)^2 / E. A large value means the observed counts are far from what "no difference" predicts.

**Why it fits our data:** our data are **counts in categories** (detected / missed) for two tests.

**Adaptation from class:** in Day 7 the expected counts came from a uniform distribution (`total / number of categories`). Here, under H0 both tests share one detection rate, so the expected counts come from the **pooled rate**: (125 + 121) / 316. With 4 cells (2 tests x detected/missed), `df = (2-1) x (2-1) = 1`.

### Step 1: Observed data (frequencies in each category)

In [ ]:
# order: FIA detected, FIA missed, IC detected, IC missed
observed = np.array([data1.sum(), len(data1)-data1.sum(), data2.sum(), len(data2)-data2.sum()])
print(observed)

### Step 2: Expected frequencies (assuming H0: both tests have the same detection rate)

In [ ]:
pooled_rate = (data1.sum() + data2.sum()) / (len(data1) + len(data2))
print('Pooled detection rate:', round(pooled_rate, 4))

expected = np.array([n*pooled_rate, n*(1-pooled_rate), n*pooled_rate, n*(1-pooled_rate)])
print(expected)

### Step 3: Chi-square statistic calculation

In [ ]:
chi_square_stat = np.sum(((observed - expected) ** 2) / expected)

### Step 4: Degrees of freedom

In [ ]:
df_chi = 1   # (rows - 1) x (columns - 1) = (2 - 1) x (2 - 1)

### Step 5: Calculate the p-value and critical value

In [ ]:
p_value_chi = 1 - stats.chi2.cdf(chi_square_stat, df_chi)  # p-value
alpha = 0.05  # Significance level
critical_value = stats.chi2.ppf(1 - alpha, df_chi)  # Critical value

### Display the results

In [ ]:
print(f'Chi-square statistic: {chi_square_stat:.2f}')
print(f'Degrees of freedom: {df_chi}')
print(f'Critical value (0.05 significance level): {critical_value:.2f}')
print(f'P-value: {p_value_chi:.4f}')

### Conclusion

In [ ]:
if chi_square_stat > critical_value:
    print('Reject the null hypothesis. FIA and IC detection rates are different.')
else:
    print('Fail to reject the null hypothesis. No significant difference between FIA and IC detection.')

### Plotting observed vs expected frequencies

In [ ]:
plt.bar(range(1, len(observed) + 1), observed, width=0.4, label='Observed', alpha=0.7)
plt.bar(np.arange(1, len(observed) + 1) + 0.4, expected, width=0.4, label='Expected', alpha=0.7)
plt.xticks(np.arange(1, 5) + 0.2, ['FIA\ndetected', 'FIA\nmissed', 'IC\ndetected', 'IC\nmissed'])
plt.xlabel('Categories')
plt.ylabel('Frequencies')
plt.title('Observed vs Expected Frequencies')
plt.legend()
plt.show()

**Reading the result:** the observed and expected bars are almost identical, which is why chi-square is small (below the critical value 3.84). This test is two-sided: it says only "different or not," not "which one is better."

---
# Technique 2 (CROSS-CHECK 1): Monte Carlo Simulation, from Day 4
**What it does:** instead of using a formula, it **simulates the experiment many times**. We assume H0 is true (both tests detect at the pooled rate), run thousands of imaginary studies, and count how often **pure chance** produces a FIA-minus-IC gap at least as big as the one we observed.

**Why it fits our data:** it makes no assumptions about the shape of the data. It works the same way as the Day 4 pi estimate: pick random numbers, check a condition, count, then divide.

**Link to Day 3:** `random.seed(...)` makes the simulation reproducible, like `np.random.seed` in Day 3.

In [ ]:
random.seed(42)   # Day 3: fixed seed so results can be reproduced

def simulate_study(num_samples, rate):
    detected = 0  # how many samples the test detected
    for i in range(num_samples):
        x = random.random()   # pick a random number from 0 to 1
        if x < rate:
            detected += 1     # the test detected dengue
    return detected

observed_diff = data1.mean() - data2.mean()
print('Observed difference (FIA - IC):', round(observed_diff, 4))

In [ ]:
num_trials = 20000
count_one_sided = 0   # trials where simulated gap >= observed gap
count_two_sided = 0   # trials where |simulated gap| >= |observed gap|
diffs = []

for t in range(num_trials):
    fia_sim = simulate_study(n, pooled_rate) / n
    ic_sim = simulate_study(n, pooled_rate) / n
    d = fia_sim - ic_sim
    diffs.append(d)
    if d >= observed_diff:
        count_one_sided += 1
    if abs(d) >= abs(observed_diff):
        count_two_sided += 1

p_mc_one = count_one_sided / num_trials
p_mc_two = count_two_sided / num_trials
print("Monte Carlo p-value (one-sided, FIA better) for", num_trials, "trials is:", p_mc_one)
print("Monte Carlo p-value (two-sided) for", num_trials, "trials is:", p_mc_two)

In [ ]:
plt.hist(diffs, bins=30, alpha=0.5, color='blue', edgecolor='black')
plt.axvline(observed_diff, color='red', linestyle='--', label='Observed gap')
plt.xlabel('FIA - IC detection rate gap (simulated under H0)')
plt.ylabel('Frequency')
plt.title('Monte Carlo: gaps produced by chance alone')
plt.legend()
plt.show()

In [ ]:
if p_mc_one < alpha:
    print('Reject the null hypothesis. A gap this large is rare by chance.')
else:
    print('Fail to reject the null hypothesis. A gap this large happens often by chance alone.')

**Reading the result:** the red line (our real gap) sits well inside the bulk of the histogram. Chance alone produces gaps this large in roughly 3 out of 10 simulated studies, so the observed gap is not unusual.

---
# Technique 3 (CROSS-CHECK 2): Two-Sample t-Test, from Day 6
**What it does:** compares the **means** of two independent groups. Here the mean of a 0/1 column is the detection rate. It also gives a **95% confidence interval** for the difference.

**Why it fits (with a caveat):** with 158 samples per group, the t-test is a good approximation even for 0/1 data. It is less natural than chi-square because t-tests are designed for continuous data. That is why it is used here as a cross-check, not as the main test.

We use the same code as Day 6 (pooled variance, `equal_var=True`).

### Step 2: Perform the two-sample t-test (pooled)

In [ ]:
t_statistic, p_value = stats.ttest_ind(data1, data2, equal_var=True)
# One-sided version for H1: FIA > IC
p_value_one = stats.ttest_ind(data1, data2, equal_var=True, alternative='greater')[1]

### Calculate confidence interval for the difference of means

In [ ]:
# mean difference
mean_diff = np.mean(data1) - np.mean(data2)

# sample sizes
n1, n2 = len(data1), len(data2)

# sample variances
s1_sq, s2_sq = np.var(data1, ddof=1), np.var(data2, ddof=1)

# pooled variance
sp_sq = ((n1 - 1) * s1_sq + (n2 - 1) * s2_sq) / (n1 + n2 - 2)

# standard error using pooled variance
std_err = np.sqrt(sp_sq * (1/n1 + 1/n2))

# degrees of freedom
df_t = n1 + n2 - 2

# 95% confidence interval
ci = stats.t.interval(0.95, df_t, loc=mean_diff, scale=std_err)

### Step 3: Display results

In [ ]:
print(f"FIA Mean (sensitivity): {np.mean(data1):.4f}")
print(f"IC Mean (sensitivity): {np.mean(data2):.4f}")
print(f"t-Statistic: {t_statistic:.2f}")
print(f"Degrees of Freedom: {df_t:.2f}")
print(f"p-Value (two-sided): {p_value:.4f}")
print(f"p-Value (one-sided, FIA > IC): {p_value_one:.4f}")
print(f"Confidence Interval of Mean Difference: [{ci[0]:.4f}, {ci[1]:.4f}]")

### Step 4: Conclusion

In [ ]:
if p_value_one < alpha:
    print("Reject the null hypothesis. FIA detects significantly better than IC.")
else:
    print("Fail to reject the null hypothesis. No significant evidence that FIA detects better than IC.")

### Step 5: Visualize the data

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(['FIA', 'IC'], [np.mean(data1), np.mean(data2)], color=['blue', 'green'], alpha=0.7)
plt.errorbar(['FIA', 'IC'], [np.mean(data1), np.mean(data2)], yerr=[np.std(data1, ddof=1), np.std(data2, ddof=1)], fmt='k_', capsize=5)
plt.title('Comparison of Sample Means (Sensitivity)')
plt.ylabel('Mean Value (detection rate)')
plt.grid(True)
plt.show()

**Reading the result:** the confidence interval for the difference includes **0**, so we cannot rule out "no difference." The error bars are large because each sample is only 0 or 1.

---
# Summary: do all three techniques agree?

In [ ]:
summary = pd.DataFrame({
    'Technique': ['Chi-square (Day 7)', 'Monte Carlo (Day 4)', 'Two-sample t-test (Day 6)'],
    'Role': ['Main test', 'Cross-check 1', 'Cross-check 2'],
    'p-value': [round(p_value_chi, 4), round(p_mc_two, 4), round(p_value, 4)],
    'Type': ['two-sided', 'two-sided', 'two-sided'],
})
summary['Decision (alpha=0.05)'] = ['Fail to reject H0' if p >= 0.05 else 'Reject H0' for p in summary['p-value']]
print(summary.to_string(index=False))
print()
print('One-sided (H1: FIA > IC):  Monte Carlo p =', round(p_mc_one, 4), ' t-test p =', round(p_value_one, 4))

## Final conclusion
All three techniques give the **same decision: fail to reject H0**.

* FIA's sensitivity (79.1%) is numerically higher than IC's (76.6%), a gap of about 2.5 percentage points.
* This gap is **not statistically significant**: chi-square p is about 0.59 (two-sided), Monte Carlo p is about 0.32 (one-sided) and the t-test one-sided p is about 0.29.
* The 95% confidence interval for the difference includes 0.
* **Interpretation:** with 158 dengue-positive samples per test, the study cannot show that FIA detects dengue better than IC. The paper itself calls the advantage "slight." Because three different methods agree, the conclusion is reliable.

**Limitations:** the rows are reconstructed from the paper's published counts (so the tests are treated as unpaired), and only NS1 sensitivity is analysed.